# C5 — Campus Library MCP (Gemini)
Run all cells top to bottom. Enter a Gemini key only when prompted.

In [ ]:
%pip -q install "fastmcp>=4.1.3,<5" "openai-agents>=0.24,<1" "python-dotenv>=1,<2"

In [ ]:
from pathlib import Path
import os, json, sys, shutil
ROOT = Path('/content/C5_Library_MCP') if Path('/content').exists() else Path.cwd() / 'C5_Library_MCP'
(ROOT / 'data').mkdir(parents=True, exist_ok=True)
(ROOT / 'tests').mkdir(parents=True, exist_ok=True)
(ROOT / 'report').mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
books = json.loads('[{"id": "B001", "title": "Python Crash Course", "author": "Eric Matthes", "topics": ["python", "programming", "beginner"], "available": true, "reserved_for": null}, {"id": "B002", "title": "Clean Code", "author": "Robert C. Martin", "topics": ["software", "programming", "design"], "available": true, "reserved_for": null}, {"id": "B003", "title": "Introduction to Algorithms", "author": "Thomas H. Cormen", "topics": ["algorithms", "data structures", "computer science"], "available": true, "reserved_for": null}, {"id": "B004", "title": "Computer Networking: A Top-Down Approach", "author": "James F. Kurose", "topics": ["networks", "internet", "communications"], "available": true, "reserved_for": null}, {"id": "B005", "title": "Operating System Concepts", "author": "Abraham Silberschatz", "topics": ["operating systems", "systems", "computer science"], "available": true, "reserved_for": null}, {"id": "B006", "title": "Fluent Python", "author": "Luciano Ramalho", "topics": ["python", "programming", "advanced"], "available": true, "reserved_for": null}, {"id": "B007", "title": "Artificial Intelligence: A Modern Approach", "author": "Stuart Russell", "topics": ["artificial intelligence", "ai", "agents"], "available": true, "reserved_for": null}, {"id": "B008", "title": "The Pragmatic Programmer", "author": "Andrew Hunt", "topics": ["software", "programming", "career"], "available": true, "reserved_for": null}, {"id": "B009", "title": "Modern Control Engineering", "author": "Katsuhiko Ogata", "topics": ["control", "engineering", "feedback"], "available": true, "reserved_for": null}, {"id": "B010", "title": "Antenna Theory: Analysis and Design", "author": "Constantine A. Balanis", "topics": ["antennas", "rf", "wireless"], "available": true, "reserved_for": null}]')
(ROOT / "data/books_original.json").write_text(json.dumps(books, indent=2) + "\n", encoding="utf-8")
shutil.copyfile(ROOT / "data/books_original.json", ROOT / "data/books.json")
(ROOT / 'requirements.txt').write_text('fastmcp>=4.1.3,<5.0\nopenai-agents>=0.24.0,<1.0\npython-dotenv>=1.0,<2.0\n', encoding="utf-8")
(ROOT / 'README.md').write_text('# Campus Library MCP (Gemini + Colab)\n\nThis project has exactly three MCP tools: `search_books` (Data), `reserve_book` (Action), and `find_and_reserve` (Orchestration). The server uses FastMCP; the agent uses the **OpenAI Agents SDK** with a **Gemini** model. The catalog is a local JSON file, not a real campus system.\n\n## Recommended: Google Colab\n\n1. Upload `C5_Library_MCP_Gemini_Colab.ipynb` to https://colab.research.google.com/.\n2. Select **Runtime > Run all**. Packages install on the Colab server.\n3. Enter a **Gemini API key** when asked (https://aistudio.google.com/apikey).\n4. Read the test and agent outputs. The notebook displays tool calls and results.\n5. Optional: Run the export cell to download the Python source project for submission.\n\nGoogle\'s free tier is subject to quotas and model availability. The default `gemini-3.5-flash-lite` is listed as free tier. You do not need an OpenAI key. The notebook stores the Gemini key only in the runtime environment, not in project files.\n\n## Local run (optional)\n\nFrom the project folder, with Python 3.11+:\n\n```bash\npython -m pip install -r requirements.txt\npython -m unittest discover -s tests -v\npython check_mcp.py\n```\n\nCopy `.env.example` to `.env`, put your Gemini key there, then run `python agent.py`. Or use `python agent.py --once "Search for Python books"`. **Never submit `.env`**.\n\n## Files\n\n- `server.py`: exactly three MCP tools exposed over stdio.\n- `library.py`: JSON catalog search and reservation operations.\n- `agent.py`: Gemini-backed OpenAI Agents SDK agent.\n- `check_mcp.py`: MCP tool discovery and sample calls, no key required.\n- `tests/test_library.py`: unit tests, including invalid input and duplicate reservation.\n- `reset_books.py`: resets catalog to the initial state.\n- `data/books.json`: editable sample catalog.\n- `report/`: reading notes and screencast outline; complete the reflection after running the agent.\n\nThe orchestration tool searches, selects the first available match, and reserves it using the same catalog operations used by the two simple tools. Success/failure returns are dictionaries with an `ok` field. The Data tool does not change state; successful reservations update `data/books.json`.\n\n## Submission\n\nThe PDF asks for a 300–400 word article/server review, source code and README, tool tests, a 5–7 minute recorded demonstration, and a 1–2 paragraph reflection. Check the written summary draft and replace generic statements with your own observations. Add your real screencast link before submitting.\n\n## Sources\n\n- https://ai.google.dev/gemini-api/docs/openai/\n- https://ai.google.dev/gemini-api/docs/pricing\n- https://openai.github.io/openai-agents-python/models/\n- https://gofastmcp.com/getting-started/quickstart\n', encoding="utf-8")
(ROOT / '.env.example').write_text('# Google AI Studio: https://aistudio.google.com/apikey\nGEMINI_API_KEY=replace_with_your_gemini_api_key\nGEMINI_MODEL=gemini-3.5-flash-lite\n', encoding="utf-8")
(ROOT / '.gitignore').write_text('.env\n.venv/\n__pycache__/\n.ipynb_checkpoints/\n*.pyc\n', encoding="utf-8")
(ROOT / 'report/reading_summary.md').write_text('# Reading summary — MCP and an existing server\n\nThe main point of the Hugging Face article by Ksenia Se is that language models need a consistent way to work with data and services outside the model itself. Before MCP, a developer often had to write a separate integration for each application. MCP provides a common protocol, so an assistant can discover tools, understand their arguments, and call them without needing a custom connection for every service. The article also makes an important distinction: MCP is the connection between an agent and its tools, not the system that decides what the agent should do.\n\nThere are three main roles in this setup. The **host** is the application running the assistant. It manages the interaction with the user and the available connections. An **MCP client** runs inside the host and communicates with an MCP server. The **server** exposes capabilities such as reading information or performing an action. In our project, the terminal agent is the host, the Agents SDK handles the client connection, and the FastMCP process serves the tools. The client discovers the tool descriptions and sends requests; the server runs the relevant code and returns the result.\n\nFor an existing implementation, I looked at the official **Filesystem MCP Server** on GitHub. It provides separate tools for reading text, writing files, listing folders, moving files and other filesystem tasks. Its README lists the inputs for each tool and explains behavior such as rejecting unsupported arguments. This is useful because the model can choose a specific operation instead of guessing what a general-purpose file command will do. The server also limits operations to allowed directories, which matters when an agent can modify files.\n\nMost of its tools have reasonably clear responsibilities. For instance, listing a directory and writing a file are separate operations. One design compromise is that some tools still have several options: `read_text_file` can return either the beginning or end of a file. That is still a focused read operation, but it needs clear validation so conflicting options are rejected. Overall, it is a good example of tools being small enough to understand, with inputs and side effects documented. It also shows why a useful tool is not only about the code working: access limits, error messages, and predictable results affect whether an agent can use it reliably.\n\n## Sources\n\n- Ksenia Se, *What Is MCP?*, Hugging Face blog: https://huggingface.co/blog/Kseniase/mcp (full article linked there: https://www.turingpost.com/p/mcp).\n- Model Context Protocol, *Filesystem MCP Server* README: https://github.com/modelcontextprotocol/servers/blob/main/src/filesystem/README.md.\n\n*Review the source articles and adjust this draft to match your own reading before submitting.*\n', encoding="utf-8")
(ROOT / 'report/reflection_notes.md').write_text("# Reflection notes — complete after testing\n\nThe assignment calls for one or two short paragraphs. Don't submit made-up problems or results. After you run the scripts, use these questions to write what happened:\n\n- Why is searching read-only, while reserving is an action?\n- Why does the combined tool call those operations in sequence?\n- Did the agent pick the expected tool for each prompt? If not, what did you change?\n- Which failure did you test (for example, booking the same book twice)? What was returned?\n- Did three tools feel like enough for this simple library, or did you find a missing feature?\n- What would you improve next (for example, cancellations, borrower IDs, concurrent access, or a real library system)?\n\nInclude only challenges and observations you actually saw in your console output.\n", encoding="utf-8")
(ROOT / 'report/video_outline.md').write_text("# Screencast outline (5–7 minutes)\n\nAim to keep the terminal visible so the tool calls and results are easy to follow.\n\n| Approx. time | What to show |\n| --- | --- |\n| 0:00–0:40 | Introduce the campus library project and identify the Data, Action, and Orchestration tools. |\n| 0:40–1:30 | Show `server.py`, `library.py`, and the catalog. Explain the single-purpose functions. |\n| 1:30–2:10 | Run `python -m unittest discover -s tests -v` and briefly mention failures covered by tests. |\n| 2:10–2:50 | Run `python check_mcp.py` to display the discovered tools and MCP calls. |\n| 2:50–4:50 | Run `python agent.py`; ask it to search, reserve by ID, and find-and-reserve. Point out the tool call / response lines. |\n| 4:50–5:40 | Try an already-reserved book or invalid ID. Open `data/books.json` to show that only successful calls changed the file. |\n| 5:40–6:30 | Explain the agent → MCP client → FastMCP server → result path, mention a real challenge, and conclude. |\n\nBefore recording, run `python reset_books.py`. Use `Demo Student` as the name. Don't display `.env`, API keys, or billing pages on screen.\n\nAdd your actual video URL to the submission after recording.\n", encoding="utf-8")
print("Project:", ROOT, "| books:", len(books))


In [ ]:
%%writefile library.py
"""Book catalog operations. No MCP or model code lives in this module."""

from __future__ import annotations

import json
import os
import tempfile
from pathlib import Path
from typing import Any


PROJECT_DIR = Path(__file__).resolve().parent
DEFAULT_CATALOG = PROJECT_DIR / "data" / "books.json"


def catalog_path() -> Path:
    """Use a separate dataset in tests when BOOKS_FILE is set."""
    configured = os.getenv("BOOKS_FILE", "").strip()
    return Path(configured).expanduser() if configured else DEFAULT_CATALOG


def read_catalog() -> list[dict[str, Any]]:
    """Read and check the catalog before using it."""
    path = catalog_path()
    try:
        with path.open("r", encoding="utf-8") as handle:
            records = json.load(handle)
    except FileNotFoundError as exc:
        raise ValueError(f"Catalog not found: {path}") from exc
    except (OSError, json.JSONDecodeError) as exc:
        raise ValueError(f"Cannot read catalog: {exc}") from exc

    if not isinstance(records, list):
        raise ValueError("Catalog must contain a list of books.")
    for record in records:
        if (
            not isinstance(record, dict)
            or not all(k in record for k in ("id", "title", "author", "topics", "available", "reserved_for"))
            or not isinstance(record["available"], bool)
            or not isinstance(record["topics"], list)
        ):
            raise ValueError("Catalog has an invalid book entry.")
    return records


def write_catalog(records: list[dict[str, Any]]) -> None:
    """Save changes using a temporary file to avoid partial JSON writes."""
    path = catalog_path()
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_name: str | None = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w", encoding="utf-8", dir=path.parent,
            prefix=".books-", suffix=".tmp", delete=False,
        ) as handle:
            temp_name = handle.name
            json.dump(records, handle, ensure_ascii=False, indent=2)
            handle.write("\n")
        os.replace(temp_name, path)
    finally:
        if temp_name and os.path.exists(temp_name):
            os.unlink(temp_name)


def search_catalog(query: str) -> dict[str, Any]:
    """Look up book titles, authors or topics; never change catalog data."""
    if not isinstance(query, str) or not query.strip():
        return {"ok": False, "error": "Enter a book title, author, or topic to search."}
    try:
        records = read_catalog()
    except ValueError as exc:
        return {"ok": False, "error": str(exc)}

    terms = query.casefold().split()
    matches = []
    for book in records:
        searchable = " ".join(
            [str(book["title"]), str(book["author"]), *map(str, book["topics"])]
        ).casefold()
        if all(term in searchable for term in terms):
            matches.append({
                "id": book["id"],
                "title": book["title"],
                "author": book["author"],
                "available": book["available"],
            })

    return {"ok": True, "query": query.strip(), "count": len(matches), "matches": matches}


def reserve_catalog_book(book_id: str, student_name: str) -> dict[str, Any]:
    """Reserve one book and persist the change to the local JSON catalog."""
    if not isinstance(book_id, str) or not book_id.strip():
        return {"ok": False, "error": "Provide a book ID."}
    if not isinstance(student_name, str) or not student_name.strip():
        return {"ok": False, "error": "Provide a name for the reservation."}
    try:
        records = read_catalog()
    except ValueError as exc:
        return {"ok": False, "error": str(exc)}

    normalized_id = book_id.strip().upper()
    for book in records:
        if str(book["id"]).upper() != normalized_id:
            continue
        if not book["available"]:
            return {"ok": False, "error": f"Book {book['id']} is already reserved."}
        book["available"] = False
        book["reserved_for"] = student_name.strip()
        try:
            write_catalog(records)
        except OSError as exc:
            return {"ok": False, "error": f"Could not save reservation: {exc}"}
        return {
            "ok": True,
            "book_id": book["id"],
            "title": book["title"],
            "reserved_for": book["reserved_for"],
            "message": "Reservation saved.",
        }
    return {"ok": False, "error": f"Book {normalized_id} was not found."}


def find_and_reserve_catalog_book(query: str, student_name: str) -> dict[str, Any]:
    """Use the read operation, then the write operation, for one request."""
    if not isinstance(student_name, str) or not student_name.strip():
        return {"ok": False, "error": "Provide a name for the reservation."}

    search_result = search_catalog(query)
    if not search_result["ok"]:
        return search_result
    if not search_result["matches"]:
        return {"ok": False, "error": f"No books matched '{query.strip()}'."}

    first_available = next(
        (book for book in search_result["matches"] if book["available"]), None
    )
    if first_available is None:
        return {"ok": False, "error": f"All books matching '{query.strip()}' are reserved."}

    reservation = reserve_catalog_book(first_available["id"], student_name)
    if not reservation["ok"]:
        return reservation
    return {
        "ok": True,
        "searched_for": query.strip(),
        "matches_found": search_result["count"],
        "reservation": reservation,
    }


In [ ]:
%%writefile server.py
"""The library's three MCP tools, served over stdio."""

import sys
from typing import Any

from fastmcp import FastMCP

from library import find_and_reserve_catalog_book, reserve_catalog_book, search_catalog


mcp = FastMCP("Campus Library")


def log_call(message: str) -> None:
    # stdout is reserved for MCP messages. Keep diagnostics on stderr.
    print(f"[MCP server] {message}", file=sys.stderr, flush=True)


@mcp.tool()
def search_books(query: str) -> dict[str, Any]:
    """Data tool. Search titles, authors and topics without changing the catalog.

    Args:
        query: A title, author, or topic, such as 'Python'.

    Returns:
        An 'ok' flag, count, and matching books with their availability;
        otherwise an 'error' string.
    """
    log_call(f"search_books(query={query!r})")
    return search_catalog(query)


@mcp.tool()
def reserve_book(book_id: str, student_name: str) -> dict[str, Any]:
    """Action tool. Reserve one available book in the local catalog.

    Args:
        book_id: Exact book ID from search results, for example 'B001'.
        student_name: Name to save with this reservation.

    Returns:
        'ok' and reservation details; or an 'error' if the book cannot
        be reserved. A successful call changes data/books.json.
    """
    log_call(f"reserve_book(book_id={book_id!r}, student_name={student_name!r})")
    return reserve_catalog_book(book_id, student_name)


@mcp.tool()
def find_and_reserve(query: str, student_name: str) -> dict[str, Any]:
    """Orchestration tool. Search books, then reserve the first available match.

    Args:
        query: Title, author, or topic to search.
        student_name: Name to save with the reservation.

    Returns:
        'ok', the search count, and reservation details; or an 'error'.
        This coordinates the same search and reservation operations used
        by the Data and Action tools.
    """
    log_call(f"find_and_reserve(query={query!r}, student_name={student_name!r})")
    return find_and_reserve_catalog_book(query, student_name)


if __name__ == "__main__":
    mcp.run(transport="stdio")


In [ ]:
%%writefile agent.py
"""MCP library agent. OpenAI Agents SDK, Gemini model."""
from __future__ import annotations

import argparse
import asyncio
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, set_tracing_disabled
from agents import OpenAIChatCompletionsModel
from agents.mcp import MCPServerStdio

ROOT = Path(__file__).resolve().parent
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
DEFAULT_MODEL = "gemini-3.5-flash-lite"


def build_agent(mcp_server):
    """Use the Agents SDK with Gemini's OpenAI-compatible endpoint."""
    load_dotenv(ROOT / ".env")
    api_key = os.getenv("GEMINI_API_KEY", "").strip()
    if not api_key:
        raise ValueError("GEMINI_API_KEY is missing. Get a key from aistudio.google.com/apikey.")
    set_tracing_disabled(True)  # No OpenAI tracing or OpenAI API key needed.
    model_client = AsyncOpenAI(api_key=api_key, base_url=GEMINI_BASE_URL)
    model = OpenAIChatCompletionsModel(
        model=os.getenv("GEMINI_MODEL", DEFAULT_MODEL),
        openai_client=model_client,
    )
    return Agent(
        name="Library Assistant",
        model=model,
        instructions=(
            "You help students search and reserve books in a sample library. "
            "Use the MCP tools for every lookup or reservation. "
            "Call search_books for searches, reserve_book for a specified book ID, "
            "and find_and_reserve to search and reserve in one request. "
            "If a student name is missing for a reservation, ask for it. "
            "Never invent book details or claim a reservation succeeded if ok is false."
        ),
        mcp_servers=[mcp_server],
    )


async def run_prompts(prompts: list[str]):
    """Connect to a real stdio MCP server and run the given prompts."""
    async with MCPServerStdio(
        name="Campus Library MCP",
        params={
            "command": sys.executable,
            "args": [str(ROOT / "server.py")],
            "cwd": str(ROOT),
        },
        cache_tools_list=True,
    ) as mcp_server:
        agent = build_agent(mcp_server)
        for prompt in prompts:
            print("\nUSER:", prompt, flush=True)
            result = await Runner.run(agent, prompt)
            for item in result.new_items:
                if item.type == "tool_call_item":
                    name = getattr(item.raw_item, "name", "MCP tool")
                    print("  TOOL CALLED:", name)
                elif item.type == "tool_call_output_item":
                    print("  TOOL RESULT:", str(item.output)[:900])
            print("AGENT:", result.final_output, flush=True)


async def interactive():
    print("Campus Library | Gemini | enter 'exit' to finish")
    while True:
        try:
            prompt = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            break
        if prompt.lower() in {"exit", "quit"}:
            break
        if prompt:
            try:
                await run_prompts([prompt])
            except Exception as exc:
                print("Error:", exc)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--once", help="Run one request")
    args = parser.parse_args()
    if args.once:
        asyncio.run(run_prompts([args.once]))
    else:
        asyncio.run(interactive())


In [ ]:
%%writefile check_mcp.py
"""Call the real MCP tools without an OpenAI API key.

This runs the FastMCP server in-process through an MCP client. It uses a
separate temporary copy of the catalog, so it won't change data/books.json.
"""

import asyncio
import json
import os
import shutil
import tempfile
from pathlib import Path

from fastmcp import Client
from server import mcp


ROOT = Path(__file__).resolve().parent


def unpack(result):
    """Get a dictionary from a FastMCP tool result."""
    if result.is_error:
        raise RuntimeError(f"MCP error: {result.content}")
    data = getattr(result, "data", None)
    if isinstance(data, dict):
        return data
    structured = getattr(result, "structured_content", None)
    if isinstance(structured, dict):
        return structured
    for block in result.content:
        if hasattr(block, "text"):
            return json.loads(block.text)
    raise RuntimeError("No JSON result returned by MCP server")


async def run_demo() -> None:
    with tempfile.TemporaryDirectory() as temp:
        demo_path = Path(temp) / "books.json"
        shutil.copyfile(ROOT / "data" / "books_original.json", demo_path)
        old_path = os.environ.get("BOOKS_FILE")
        os.environ["BOOKS_FILE"] = str(demo_path)
        try:
            async with Client(mcp) as client:
                tools = await client.list_tools()
                names = sorted(tool.name for tool in tools)
                expected = sorted(("search_books", "reserve_book", "find_and_reserve"))
                assert names == expected, f"Expected exactly 3 tools, got: {names}"
                print("Discovered tools:", ", ".join(names))

                calls = [
                    ("search_books", {"query": "Python"}),
                    ("reserve_book", {"book_id": "B001", "student_name": "Demo Student"}),
                    ("reserve_book", {"book_id": "B001", "student_name": "Demo Student"}),
                    ("find_and_reserve", {"query": "algorithms", "student_name": "Demo Student"}),
                    ("find_and_reserve", {"query": "not-a-real-topic", "student_name": "Demo Student"}),
                ]
                for name, arguments in calls:
                    response = unpack(await client.call_tool(name, arguments))
                    print(f"\n{name}({arguments}):")
                    print(json.dumps(response, indent=2))
                print("\nMCP checks finished. The original catalog was not modified.")
        finally:
            if old_path is None:
                os.environ.pop("BOOKS_FILE", None)
            else:
                os.environ["BOOKS_FILE"] = old_path


if __name__ == "__main__":
    asyncio.run(run_demo())


In [ ]:
%%writefile reset_books.py
"""Put the sample catalog back to its original state for another demo."""

import shutil
from pathlib import Path


base = Path(__file__).resolve().parent / "data"
source = base / "books_original.json"
target = base / "books.json"

if __name__ == "__main__":
    shutil.copyfile(source, target)
    print("Sample book data reset. All books are available again.")


In [ ]:
%%writefile tests/test_library.py
"""Tests for the catalog logic. No API key or third-party packages needed."""

import json
import os
import shutil
import tempfile
import unittest
from pathlib import Path
from unittest.mock import patch

from library import (
    find_and_reserve_catalog_book, read_catalog,
    reserve_catalog_book, search_catalog,
)


ROOT = Path(__file__).resolve().parents[1]


class LibraryTests(unittest.TestCase):
    def setUp(self):
        self.temp = tempfile.TemporaryDirectory()
        self.addCleanup(self.temp.cleanup)
        self.path = Path(self.temp.name) / "books.json"
        shutil.copyfile(ROOT / "data" / "books_original.json", self.path)
        env_patch = patch.dict(os.environ, {"BOOKS_FILE": str(self.path)})
        env_patch.start()
        self.addCleanup(env_patch.stop)

    def test_search_works_and_does_not_modify_file(self):
        before = self.path.read_bytes()
        answer = search_catalog("python")
        self.assertTrue(answer["ok"])
        self.assertGreaterEqual(answer["count"], 1)
        self.assertEqual(self.path.read_bytes(), before)

    def test_empty_search_is_rejected(self):
        self.assertFalse(search_catalog("   ")["ok"])

    def test_search_matches_author_or_topic(self):
        self.assertGreater(search_catalog("martin")["count"], 0)
        self.assertGreater(search_catalog("networks")["count"], 0)

    def test_reservation_saves_to_disk(self):
        reply = reserve_catalog_book("B001", "Test Student")
        self.assertTrue(reply["ok"])
        book = next(b for b in read_catalog() if b["id"] == "B001")
        self.assertFalse(book["available"])
        self.assertEqual(book["reserved_for"], "Test Student")

    def test_duplicate_reservation_is_rejected(self):
        reserve_catalog_book("B001", "First Student")
        answer = reserve_catalog_book("B001", "Second Student")
        self.assertFalse(answer["ok"])
        self.assertIn("already reserved", answer["error"])
        book = next(b for b in read_catalog() if b["id"] == "B001")
        self.assertEqual(book["reserved_for"], "First Student")

    def test_unknown_id_is_rejected(self):
        self.assertIn("not found", reserve_catalog_book("B999", "Test Student")["error"])

    def test_missing_reservation_name_is_rejected(self):
        self.assertFalse(reserve_catalog_book("B001", "")["ok"])

    def test_orchestration_completes_both_steps(self):
        answer = find_and_reserve_catalog_book("algorithms", "Test Student")
        self.assertTrue(answer["ok"])
        self.assertGreaterEqual(answer["matches_found"], 1)
        self.assertEqual(answer["reservation"]["reserved_for"], "Test Student")
        book_id = answer["reservation"]["book_id"]
        self.assertFalse(next(b for b in read_catalog() if b["id"] == book_id)["available"])

    def test_orchestration_handles_no_match(self):
        self.assertFalse(find_and_reserve_catalog_book("xyz-no-match", "Test Student")["ok"])

    def test_orchestration_handles_all_reserved(self):
        reserve_catalog_book("B001", "First Student")
        response = find_and_reserve_catalog_book("Python Crash Course", "Second Student")
        self.assertFalse(response["ok"])
        self.assertIn("reserved", response["error"])

    def test_missing_catalog_returns_error(self):
        self.path.unlink()
        answer = search_catalog("Python")
        self.assertFalse(answer["ok"])
        self.assertIn("not found", answer["error"])

    def test_corrupt_catalog_returns_error(self):
        self.path.write_text("{broken json", encoding="utf-8")
        self.assertFalse(search_catalog("Python")["ok"])


if __name__ == "__main__":
    unittest.main()


### Tests (no API key)

In [ ]:
import subprocess
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=ROOT, check=True)
subprocess.run([sys.executable, 'check_mcp.py'], cwd=ROOT, check=True)

### Gemini agent

In [ ]:
from getpass import getpass
key = os.environ.get('GEMINI_API_KEY', '').strip()
if not key:
    try:
        from google.colab import userdata
        key = (userdata.get('GEMINI_API_KEY') or '').strip()
    except Exception:
        pass
if not key:
    key = getpass('Gemini API key (https://aistudio.google.com/apikey): ').strip()
if not key:
    raise ValueError('GEMINI_API_KEY is required for the agent demo.')
os.environ['GEMINI_API_KEY'] = key
os.environ['GEMINI_MODEL'] = 'gemini-3.5-flash-lite'
print('Gemini key loaded (not saved in project files).')

In [ ]:
from agent import run_prompts
# Reset catalog before the recorded demo.
shutil.copyfile(ROOT / 'data/books_original.json', ROOT / 'data/books.json')
await run_prompts([
    'Search for Python books. Do not reserve anything.',
    'Reserve book B001 for Demo Student.',
    'Find an available algorithms book and reserve it for Demo Student.',
])

### Optional: failure case and export

In [ ]:
await run_prompts(['Try to reserve book B001 again for Demo Student. Explain the error.'])

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
out_zip = ROOT.parent / 'C5_Library_MCP_Submission_Source.zip'
paths = [
    'library.py', 'server.py', 'agent.py', 'check_mcp.py', 'reset_books.py',
    'tests/test_library.py', 'data/books.json', 'data/books_original.json',
    'requirements.txt', 'README.md', '.env.example', '.gitignore',
    'report/reading_summary.md', 'report/reflection_notes.md', 'report/video_outline.md',
]
with ZipFile(out_zip, 'w', ZIP_DEFLATED) as z:
    for relative in paths:
        if (ROOT / relative).exists():
            z.write(ROOT / relative, arcname=f'C5_Library_MCP/{relative}')
print('Source project ready:', out_zip)
# To download from Colab, run:
# from google.colab import files
# files.download(str(out_zip))